# 05 Import Hugging Face GitHub Code Review Dataset

Purpose: download `ronantakizawa/github-codereview` and transform it into the local PR suggestion coverage format.

This dataset is useful for language coverage and calibration, but it is a weak fit for exact suggestion-landed labels. It records review comments and later changed code, not guaranteed exact GitHub suggestion patches.

Default behavior is conservative: keep positive rows only when the reviewer comment contains a fenced code block, because those rows look most like small code suggestions.


## Setup

Install the ML requirements first if this notebook cannot import `datasets`:

```bash
uv sync --locked --extra notebooks
```


In [1]:
from __future__ import annotations

import csv
import difflib
import hashlib
import json
import re
from collections import Counter
from pathlib import Path

try:
    from datasets import load_dataset
except ImportError as exc:
    raise ImportError('Install notebook dependencies first: uv sync --locked --extra notebooks') from exc


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'pr_suggestion_metrics').is_dir():
            return candidate
    raise RuntimeError(f'Could not find project root from {start}')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'external' / 'github_codereview' / 'dataset'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = 'ronantakizawa/github-codereview'
SPLIT = 'train'
REFRESH_IMPORT = True  # keep False to reuse local files; set True to download/stream HF again
TARGET_ROWS = 2500  # accepted transformed rows, not raw HF rows
MAX_SCANNED_ROWS = 50000  # hard stop for raw streamed rows when REFRESH_IMPORT=True
PROGRESS_EVERY = 250
ALLOW_PARTIAL_ON_INTERRUPT = True
LANGUAGE_ALLOWLIST = None  # example: {'Python', 'Go', 'JavaScript'}
INCLUDE_NEGATIVES = False
REQUIRE_CODE_BLOCK_FOR_POSITIVES = True

DATASET_JSONL = OUTPUT_DIR / 'dataset.jsonl'
LABELS_CSV = OUTPUT_DIR / 'labels.csv'
LLM_LABELS_JSONL = OUTPUT_DIR / 'llm_labels.jsonl'
README_PATH = OUTPUT_DIR / 'README.md'

OUTPUT_DIR


PosixPath('/Users/I551270/Documents/GitHub/pipeline-fl-control-plane/ml/data/external/github_codereview/dataset')

## Transform Helpers

The local evaluator expects diffs. The Hugging Face dataset gives `before_code`, `after_code`, and `reviewer_comment`, so this notebook creates synthetic unified diffs for evaluation.


In [2]:
CODE_BLOCK_PATTERN = re.compile(r'```(?:[^\n`]*)\n(.*?)```', re.DOTALL)
LABEL_TO_PERCENTAGE = {'0%': 0, 'partial': 40, 'mostly': 80, '100%': 100}
DIFF_PATH_PATTERN = re.compile(r'^diff --git a/(.*?) b/(.*)$')
OLD_FILE_PATTERN = re.compile(r'^---\s+(?:a/)?(.+)$')
NEW_FILE_PATTERN = re.compile(r'^\+\+\+\s+(?:b/)?(.+)$')
HUNK_PATTERN = re.compile(r'^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@')


def stable_id(*parts: object) -> str:
    payload = '|'.join(str(part) for part in parts)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()[:16]


def split_code_lines(code: str | None) -> list[str]:
    if not code:
        return []
    return code.splitlines()


def unified_diff(before_code: str, after_code: str, file_path: str) -> str:
    before_lines = [line + '\n' for line in split_code_lines(before_code)]
    after_lines = [line + '\n' for line in split_code_lines(after_code)]
    return ''.join(
        difflib.unified_diff(
            before_lines,
            after_lines,
            fromfile=f'a/{file_path}',
            tofile=f'b/{file_path}',
            lineterm='\n',
        )
    )


def added_only_diff(code: str, file_path: str) -> str:
    lines = split_code_lines(code)
    body = '\n'.join(f'+{line}' for line in lines)
    return f'--- a/{file_path}\n+++ b/{file_path}\n@@\n{body}' if body else ''


def extract_code_blocks(comment: str | None) -> list[str]:
    if not comment:
        return []
    return [block.strip('\n') for block in CODE_BLOCK_PATTERN.findall(comment) if block.strip()]


def diff_stats(diff_text: str) -> dict[str, object]:
    files: set[str] = set()
    added_lines = 0
    removed_lines = 0
    hunk_count = 0
    for line in diff_text.splitlines():
        git_match = DIFF_PATH_PATTERN.match(line)
        if git_match:
            files.add(git_match.group(2))
            continue
        new_file_match = NEW_FILE_PATTERN.match(line)
        if new_file_match and new_file_match.group(1) != '/dev/null':
            files.add(new_file_match.group(1))
            continue
        old_file_match = OLD_FILE_PATTERN.match(line)
        if old_file_match and old_file_match.group(1) != '/dev/null':
            files.add(old_file_match.group(1))
            continue
        if HUNK_PATTERN.match(line) or line.startswith('@@'):
            hunk_count += 1
            continue
        if line.startswith('+++') or line.startswith('---'):
            continue
        if line.startswith('+'):
            added_lines += 1
        elif line.startswith('-'):
            removed_lines += 1
    return {
        'file_count': len(files),
        'added_lines': added_lines,
        'removed_lines': removed_lines,
        'hunk_count': hunk_count,
        'files': sorted(files),
    }


def normalized_added_lines(diff_text: str) -> list[str]:
    lines = []
    for line in diff_text.splitlines():
        if line.startswith('+++') or line.startswith('---') or line.startswith('@@'):
            continue
        if line.startswith('+'):
            normalized = ' '.join(line[1:].strip().split())
            if normalized:
                lines.append(normalized)
    return lines


def line_overlap_ratio(suggested_diff: str, landed_diff: str) -> float:
    suggested = normalized_added_lines(suggested_diff)
    landed = Counter(normalized_added_lines(landed_diff))
    if not suggested:
        return 0.0
    matched = 0
    for line in suggested:
        if landed[line] > 0:
            matched += 1
            landed[line] -= 1
    return matched / len(suggested)


def label_for_row(row: dict, has_comment_code: bool, overlap: float) -> tuple[str, int, str, str]:
    if row.get('is_negative'):
        return '0%', 0, 'HF negative example: no reviewer issue.', 'hf_negative'
    if not has_comment_code:
        raise ValueError('Positive rows must have an explicit reviewer code block; refusing circular synthetic labels.')
    if overlap >= 0.95:
        return '100%', 100, 'Weak label: reviewer code block appears almost exactly in the after-change diff.', 'weak_line_overlap'
    if overlap >= 0.60:
        return 'mostly', 80, 'Weak label: most reviewer code-block lines appear in the after-change diff.', 'weak_line_overlap'
    if overlap > 0:
        return 'partial', 40, 'Weak label: some reviewer code-block lines appear in the after-change diff.', 'weak_line_overlap'
    return '0%', 0, 'Weak label: reviewer code block does not appear in the after-change diff.', 'weak_line_overlap'


def row_to_local_example(row: dict) -> tuple[dict, dict, dict] | None:
    file_path = row.get('file_path') or 'unknown.txt'
    repo_name = row.get('repo_name') or 'unknown/unknown'
    pr_number = row.get('pr_number') or 0
    language = row.get('language') or row.get('repo_language') or 'unknown'

    if LANGUAGE_ALLOWLIST is not None and language not in LANGUAGE_ALLOWLIST:
        return None
    if row.get('is_negative') and not INCLUDE_NEGATIVES:
        return None

    before_code = row.get('before_code') or ''
    after_code = row.get('after_code') or ''
    landed_diff = unified_diff(before_code, after_code, file_path)
    comment_code_blocks = extract_code_blocks(row.get('reviewer_comment'))

    if comment_code_blocks:
        suggested_diff = added_only_diff(comment_code_blocks[0], file_path)
        suggestion_source = 'hf_review_comment_code_block'
    elif row.get('is_negative'):
        suggested_diff = ''
        suggestion_source = 'hf_negative_no_suggestion'
    elif REQUIRE_CODE_BLOCK_FOR_POSITIVES:
        return None
    else:
        raise ValueError('Refusing to create suggested_diff from landed_diff for a positive row.')

    if not landed_diff.strip() and not row.get('is_negative'):
        return None

    overlap = line_overlap_ratio(suggested_diff, landed_diff)
    label, expected_percentage, label_notes, label_source = label_for_row(row, bool(comment_code_blocks), overlap)
    suggested_stats = diff_stats(suggested_diff)
    landed_stats = diff_stats(landed_diff)
    example_id = stable_id(DATASET_NAME, SPLIT, repo_name, pr_number, file_path, row.get('comment_line'), suggested_diff, landed_diff)
    pr_url = f'https://github.com/{repo_name}/pull/{pr_number}' if repo_name != 'unknown/unknown' and pr_number else ''

    dataset_row = {
        'example_id': example_id,
        'pr_url': pr_url,
        'suggested_diff': suggested_diff,
        'landed_diff': landed_diff,
        'repo': repo_name,
        'file_path': file_path,
        'language': language,
        'suggestion_source': suggestion_source,
        'label': label,
        'expected_landed_percentage': expected_percentage,
        'suggested_stats': suggested_stats,
        'landed_stats': landed_stats,
        'deterministic_landed_estimate': expected_percentage,
        'file_overlap_ratio': 1.0 if file_path else 0.0,
        'changed_line_overlap_ratio': round(overlap, 4),
        'metadata': {
            'source_dataset': DATASET_NAME,
            'source_split': SPLIT,
            'language': language,
            'file_path': file_path,
            'comment_type': row.get('comment_type'),
            'quality_score': row.get('quality_score'),
            'is_negative': row.get('is_negative'),
            'weak_label': True,
            'label_source': label_source,
        },
    }
    label_row = {
        'example_id': example_id,
        'label': label,
        'expected_landed_percentage': expected_percentage,
        'label_notes': label_notes,
        'suggested_label': label,
        'suggested_percentage': expected_percentage,
        'suggested_rationale': label_notes,
        'pr_url': pr_url,
        'repo': repo_name,
        'suggestion_source': suggestion_source,
        'deterministic_landed_estimate': expected_percentage,
        'file_overlap_ratio': 1.0 if file_path else 0.0,
        'changed_line_overlap_ratio': round(overlap, 4),
        'suggested_files': file_path,
        'landed_files': file_path,
    }
    llm_label_row = {
        'example_id': example_id,
        'label': label,
        'expected_landed_percentage': expected_percentage,
        'rationale': label_notes,
        'source_dataset': DATASET_NAME,
        'label_source': label_source,
        'weak_label': True,
    }
    return dataset_row, label_row, llm_label_row


## Download And Transform

This uses Hugging Face streaming by default, so `MAX_ROWS` controls how much you pull locally.


In [3]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]


def read_csv_rows(path: Path) -> list[dict]:
    with path.open(newline='', encoding='utf-8') as stream:
        return list(csv.DictReader(stream))


def local_dataset_exists() -> bool:
    return DATASET_JSONL.exists() and LABELS_CSV.exists() and LLM_LABELS_JSONL.exists()


dataset_rows = []
label_rows = []
llm_label_rows = []
skipped = Counter()
seen_ids = set()
raw_rows_scanned = 0
loaded_from_local_files = False

if local_dataset_exists() and not REFRESH_IMPORT:
    dataset_rows = read_jsonl(DATASET_JSONL)
    label_rows = read_csv_rows(LABELS_CSV)
    llm_label_rows = read_jsonl(LLM_LABELS_JSONL)
    seen_ids = {row['example_id'] for row in dataset_rows}
    loaded_from_local_files = True
    print('Loaded existing local dataset files. Set REFRESH_IMPORT=True to download from Hugging Face again.')
else:
    stream = load_dataset(DATASET_NAME, split=SPLIT, streaming=True)

    try:
        for index, row in enumerate(stream, start=1):
            raw_rows_scanned = index
            if MAX_SCANNED_ROWS is not None and raw_rows_scanned > MAX_SCANNED_ROWS:
                skipped['max_scanned_rows_reached'] += 1
                break
            if TARGET_ROWS is not None and len(dataset_rows) >= TARGET_ROWS:
                break

            transformed = row_to_local_example(row)
            if transformed is None:
                if row.get('is_negative') and not INCLUDE_NEGATIVES:
                    skipped['negative_disabled'] += 1
                elif not extract_code_blocks(row.get('reviewer_comment')) and REQUIRE_CODE_BLOCK_FOR_POSITIVES:
                    skipped['positive_without_code_block'] += 1
                else:
                    skipped['other'] += 1
            else:
                dataset_row, label_row, llm_label_row = transformed
                if dataset_row['example_id'] in seen_ids:
                    skipped['duplicate'] += 1
                else:
                    seen_ids.add(dataset_row['example_id'])
                    dataset_rows.append(dataset_row)
                    label_rows.append(label_row)
                    llm_label_rows.append(llm_label_row)

            if PROGRESS_EVERY and raw_rows_scanned % PROGRESS_EVERY == 0:
                print(f'scanned={raw_rows_scanned:,} kept={len(dataset_rows):,} skipped={sum(skipped.values()):,}')
    except KeyboardInterrupt:
        print(f'Interrupted after scanned={raw_rows_scanned:,}, kept={len(dataset_rows):,}.')
        if not ALLOW_PARTIAL_ON_INTERRUPT:
            raise

print('loaded_from_local_files:', loaded_from_local_files)
print('raw rows scanned:', raw_rows_scanned)
print('kept rows:', len(dataset_rows))
print('skipped:', skipped)
print('labels:', Counter(row['label'] for row in label_rows))
print('languages:', Counter(row.get('metadata', {}).get('language') for row in dataset_rows).most_common(20))
dataset_rows[:2]


scanned=250 kept=57 skipped=193
scanned=500 kept=80 skipped=420
scanned=750 kept=122 skipped=628
scanned=1,000 kept=219 skipped=781
scanned=1,250 kept=231 skipped=1,019
scanned=1,500 kept=345 skipped=1,155
scanned=1,750 kept=457 skipped=1,293
scanned=2,000 kept=479 skipped=1,521
scanned=2,250 kept=538 skipped=1,712
scanned=2,500 kept=665 skipped=1,835
scanned=2,750 kept=799 skipped=1,951
scanned=3,000 kept=891 skipped=2,109
scanned=3,250 kept=926 skipped=2,324
scanned=3,500 kept=939 skipped=2,561
scanned=3,750 kept=995 skipped=2,755
scanned=4,000 kept=1,032 skipped=2,968
scanned=4,250 kept=1,121 skipped=3,129
scanned=4,500 kept=1,138 skipped=3,362
scanned=4,750 kept=1,150 skipped=3,600
scanned=5,000 kept=1,216 skipped=3,784
scanned=5,250 kept=1,236 skipped=4,014
scanned=5,500 kept=1,356 skipped=4,144
scanned=5,750 kept=1,495 skipped=4,255
scanned=6,000 kept=1,540 skipped=4,460
scanned=6,250 kept=1,569 skipped=4,681
scanned=6,500 kept=1,595 skipped=4,905
scanned=6,750 kept=1,611 skipped

[{'example_id': '4cd17d7210f1d44c',
  'pr_url': 'https://github.com/4ian/GDevelop/pull/6970',
  'suggested_diff': '--- a/GDJS/tests/tests/Extensions/testspriteruntimeobject.js\n+++ b/GDJS/tests/tests/Extensions/testspriteruntimeobject.js\n@@\n+   this._customHeight = height;',
  'landed_diff': '--- a/GDJS/tests/tests/Extensions/testspriteruntimeobject.js\n+++ b/GDJS/tests/tests/Extensions/testspriteruntimeobject.js\n@@ -23,7 +23,7 @@\n   }\n \n   setHeight(height) {\n-    return this._customHeight = height;\n+    this._customHeight = height;\n   }\n \n   getCenterX() {\n',
  'repo': '4ian/GDevelop',
  'file_path': 'GDJS/tests/tests/Extensions/testspriteruntimeobject.js',
  'language': 'JavaScript',
  'suggestion_source': 'hf_review_comment_code_block',
  'label': '100%',
  'expected_landed_percentage': 100,
  'suggested_stats': {'file_count': 1,
   'added_lines': 1,
   'removed_lines': 0,
   'hunk_count': 1,
   'files': ['GDJS/tests/tests/Extensions/testspriteruntimeobject.js']},
  'la

## Write Local Dataset Files

Writes a dataset compatible with `evaluate_metrics.py`. These labels are weak labels, so keep them separate from the hand-labeled project dataset.


In [4]:
if not dataset_rows:
    raise ValueError('No rows to write. Keep REFRESH_IMPORT=False to reuse existing files or increase MAX_SCANNED_ROWS/TARGET_ROWS.')

with DATASET_JSONL.open('w', encoding='utf-8') as stream:
    for row in dataset_rows:
        stream.write(json.dumps(row, ensure_ascii=False) + '\n')

label_fieldnames = [
    'example_id',
    'label',
    'expected_landed_percentage',
    'label_notes',
    'suggested_label',
    'suggested_percentage',
    'suggested_rationale',
    'pr_url',
    'repo',
    'suggestion_source',
    'deterministic_landed_estimate',
    'file_overlap_ratio',
    'changed_line_overlap_ratio',
    'suggested_files',
    'landed_files',
]
with LABELS_CSV.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=label_fieldnames)
    writer.writeheader()
    writer.writerows(label_rows)

with LLM_LABELS_JSONL.open('w', encoding='utf-8') as stream:
    for row in llm_label_rows:
        stream.write(json.dumps(row, ensure_ascii=False) + '\n')

README_PATH.write_text(
    '\n'.join([
        '# Hugging Face GitHub Code Review Import',
        '',
        f'Source dataset: {DATASET_NAME}',
        f'Split: {SPLIT}',
        f'Refresh import: {REFRESH_IMPORT}',
        f'Target accepted rows: {TARGET_ROWS}',
        f'Max raw rows scanned: {MAX_SCANNED_ROWS}',
        f'Actual raw rows scanned: {raw_rows_scanned}',
        f'Actual kept rows: {len(dataset_rows)}',
        '',
        'Labels are weak labels derived from review-response metadata.',
        'Keep this dataset separate from the hand-labeled project dataset.',
        '',
    ]),
    encoding='utf-8',
)

DATASET_JSONL, LABELS_CSV, LLM_LABELS_JSONL, README_PATH


(PosixPath('/Users/I551270/Documents/GitHub/pipeline-fl-control-plane/ml/data/external/github_codereview/dataset/dataset.jsonl'),
 PosixPath('/Users/I551270/Documents/GitHub/pipeline-fl-control-plane/ml/data/external/github_codereview/dataset/labels.csv'),
 PosixPath('/Users/I551270/Documents/GitHub/pipeline-fl-control-plane/ml/data/external/github_codereview/dataset/llm_labels.jsonl'),
 PosixPath('/Users/I551270/Documents/GitHub/pipeline-fl-control-plane/ml/data/external/github_codereview/dataset/README.md'))

## Optional: Run Evaluator On Imported Dataset

This validates that the transformed files match the local evaluator schema.


In [5]:
import subprocess
import sys

scores_path = OUTPUT_DIR.parent / 'metric_scores.csv'
cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'src' / 'pr_suggestion_metrics' / 'evaluate_metrics.py'),
    '--dataset-dir',
    str(OUTPUT_DIR),
    '--output',
    str(scores_path),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
scores_path


Running: /Users/I551270/.cache/uv/builds-v0/.tmpBjUHZT/bin/python /Users/I551270/Documents/GitHub/pipeline-fl-control-plane/ml/src/pr_suggestion_metrics/evaluate_metrics.py --dataset-dir /Users/I551270/Documents/GitHub/pipeline-fl-control-plane/ml/data/external/github_codereview/dataset --output /Users/I551270/Documents/GitHub/pipeline-fl-control-plane/ml/data/external/github_codereview/metric_scores.csv


<unknown>:13: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.


examples: 2500
baseline MAE: 0.00
new metric MAE: 25.81

baseline deterministic_landed_estimate
accuracy: 1.000
confusion actual -> predicted
actual,0%,partial,mostly,100%
0%,994,0,0,0
partial,0,419,0,0
mostly,0,0,126,0
100%,0,0,0,961
per-class precision recall f1
0%: precision=1.000 recall=1.000 f1=1.000
partial: precision=1.000 recall=1.000 f1=1.000
mostly: precision=1.000 recall=1.000 f1=1.000
100%: precision=1.000 recall=1.000 f1=1.000

new clone-style metric stack
accuracy: 0.638
confusion actual -> predicted
actual,0%,partial,mostly,100%
0%,280,91,607,16
partial,8,239,170,2
mostly,3,1,114,8
100%,0,0,0,961
per-class precision recall f1
0%: precision=0.962 recall=0.282 f1=0.436
partial: precision=0.722 recall=0.570 f1=0.637
mostly: precision=0.128 recall=0.905 f1=0.224
100%: precision=0.974 recall=1.000 f1=0.987

wrote per-example scores: /Users/I551270/Documents/GitHub/pipeline-fl-control-plane/ml/data/external/github_codereview/metric_scores.csv


PosixPath('/Users/I551270/Documents/GitHub/pipeline-fl-control-plane/ml/data/external/github_codereview/metric_scores.csv')